# Code Broker: Multi-Agent Code Assessment System

This notebook implements a sophisticated multi-agent system for comprehensive code analysis using Google's ADK (Agent Development Kit).

## Features
- 🔍 **Code Description Generation**: Analyzes code structure and functionality
- ✅ **Correctness Assessment**: Evaluates functionality, security, and efficiency
- 🎨 **Style Assessment**: Checks readability, maintainability, and best practices
- 💡 **Improvement Recommendations**: Suggests actionable enhancements
- 📊 **Comprehensive Reporting**: Compiles all assessments into a professional report

## Requirements
- Google ADK
- Python 3.10+
- pylint (for code linting)
- Git (for repository cloning)

## 1️⃣ Imports and Dependencies

Import all necessary libraries for the multi-agent system:
- **Google ADK**: Agent framework components
- **asyncio**: Asynchronous execution
- **IPython**: Notebook display utilities

In [1]:
import os
import re
import shutil
import tempfile
import asyncio
import time
from pathlib import Path
from IPython.display import display, Markdown
from google.adk.models.google_llm import Gemini
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent
from google.adk.runners import Runner
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool, ToolContext
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import load_memory, preload_memory
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.apps.app import App, ResumabilityConfig
from google.genai import types
from dotenv import load_dotenv

## 2️⃣ Environment Initialization

This function loads API credentials from your `.env` file:
- `GOOGLE_API_KEY`: Required for Gemini model access
- `GITHUB_TOKEN`: Optional, for private repository access

**Note**: Create a `.env` file in your project root with these variables.

In [2]:
async def initialize_adk_model():
    """
    Initializes the Google ADK environment with required API credentials.
    
    Raises:
        ValueError: If GOOGLE_API_KEY or GITHUB_TOKEN are not set
        
    Returns:
        dict: Configuration dictionary with API credentials
    """
    load_dotenv()
    api_key = os.getenv("GOOGLE_API_KEY")
    github_token = os.getenv("GITHUB_TOKEN")
    
    os.environ["GOOGLE_API_KEY"] = api_key
    os.environ["GITHUB_TOKEN"] = github_token
    
    if not api_key:
        raise ValueError("GOOGLE_API_KEY environment variable not set.")
    if not github_token:
        raise ValueError("GITHUB_TOKEN environment variable not set.")
    
    return {"api_key": api_key, "github_token": github_token}

## 3️⃣ API Retry Configuration

Configure exponential backoff retry behavior to handle:
- Rate limiting (HTTP 429)
- Transient server errors (HTTP 500, 503, 504)
- Network timeouts

In [3]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Exponential backoff multiplier
    initial_delay=1,  # Initial delay in seconds
    http_status_codes=[429, 500, 503, 504],  # Retry on these errors
)

## 4️⃣ File and Repository Utility Functions

These tools enable agents to:
- Read individual files or entire directories
- Clone and analyze GitHub repositories
- Run pylint for code quality assessment
- Clean up temporary directories

In [4]:
def read_file(file_path: str) -> str:
    """Reads and returns file content with error handling."""
    if not os.path.exists(file_path):
        return f"Error: File not found at {file_path}"
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    except Exception as e:
        return f"Error reading file: {e}"


def read_directory_files(directory_path: str) -> dict[str, str]:
    """Recursively reads all files in a directory, excluding .git."""
    if not os.path.isdir(directory_path):
        return {"error": f"Error: Directory not found at {directory_path}"}
    
    file_contents = {}
    for root, _, files in os.walk(directory_path):
        for file in files:
            file_path = os.path.join(root, file)
            if '.git' in file_path.split(os.sep):
                continue
            file_contents[file_path] = read_file(file_path)
    return file_contents


async def read_github_repository(repo_url: str) -> dict:
    """Clones a GitHub repository and reads all its files."""
    print(f"Debug: Cloning repository: {repo_url}")
    
    if not re.match(r"https://github\.com/([^/]+)/([^/]+)", repo_url):
        return {"error": "Invalid GitHub repository URL provided."}

    temp_dir = tempfile.mkdtemp()
    try:
        command = f"git clone --depth 1 {repo_url} ."
        process = await asyncio.create_subprocess_shell(
            command,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
            cwd=temp_dir
        )
        stdout, stderr = await process.communicate()

        if process.returncode != 0:
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)
            return {"error": f"Failed to clone repository. Error: {stderr.decode()}"}

        file_contents = read_directory_files(temp_dir)
        return {"file_contents": file_contents, "temp_dir": temp_dir}
    except Exception as e:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
        return {"error": f"An unexpected error occurred: {e}"}


def cleanup_temp_directory(directory_path: str) -> dict:
    """Safely removes a temporary directory and its contents."""
    if not os.path.isdir(directory_path):
        return {"error": f"Error: Directory not found at {directory_path}"}
    try:
        shutil.rmtree(directory_path)
        return {"success": f"Successfully removed directory: {directory_path}"}
    except Exception as e:
        return {"error": f"Error removing directory {directory_path}: {e}"}


async def get_linting_score(path: str) -> dict:
    """Runs pylint and returns a quality score (0-100%)."""
    if not os.path.exists(path):
        return {"error": f"Error: Path not found at {path}"}
    
    if os.path.isfile(path) and not path.endswith('.py'):
        return {"error": "Error: Not a Python file."}

    try:
        absolute_path = os.path.abspath(path)
        command = f"pylint {absolute_path}"
        
        process = await asyncio.create_subprocess_shell(
            command,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE
        )
        stdout, stderr = await process.communicate()
        pylint_output = stdout.decode()
        pylint_error = stderr.decode()

        if process.returncode != 0 and "No such file or directory" in pylint_error:
            return {"error": "Pylint not installed. Install with: pip install pylint"}
        
        match = re.search(r"Your code has been rated at ([-+]?\d*\.\d+|\d+)/10", pylint_output)
        if match:
            score_out_of_10 = float(match.group(1))
            return {"linting_score": score_out_of_10 * 10}
        else:
            return {
                "linting_score": 0.0,
                "message": "Pylint ran but no score found",
                "pylint_output": pylint_output[:500]
            }
    except Exception as e:
        return {"error": f"Linting error: {e}"}

## 5️⃣ Specialized Assessment Agents

### Agent Architecture:
1. **Correctness Assessor**: Evaluates functionality, security, efficiency, and tests
2. **Style Assessor**: Checks readability, maintainability, and best practices
3. **Description Generator**: Creates comprehensive code documentation
4. **Improvement Recommender**: Suggests actionable enhancements
5. **Report Generator**: Orchestrates all agents and compiles final report

Each agent has access to specialized tools for code analysis.

In [5]:
# AGENT 1: Correctness Assessor
correctness_assessor = LlmAgent(
    model=Gemini(model="gemini-3.5-flash-lite", retry_options=retry_config),
    name="correctness_assessor",
    description="Evaluates code functionality, security, and efficiency",
    instruction="""
        You are a code correctness expert. Evaluate the code on:
        1. **Functionality**: Error handling, edge cases, requirement delivery
        2. **Security**: Vulnerabilities, exploits, insecure practices
        3. **Resource Efficiency**: CPU, memory, resource management
        4. **Test Coverage**: Presence and quality of tests

        Provide analysis (2-3 sentences) and score (0-100%) for each.
        Present in structured Markdown with overall score.
        """,
    tools=[
        FunctionTool(read_file),
        FunctionTool(read_directory_files),
        FunctionTool(read_github_repository),
        FunctionTool(get_linting_score),
        FunctionTool(cleanup_temp_directory),
    ],
)

# AGENT 2: Style Assessor
style_assessor = LlmAgent(
    model=Gemini(model="gemini-3.5-flash-lite", retry_options=retry_config),
    name="style_assessor",
    description="Evaluates code style and maintainability",
    instruction="""
        You are a code style expert. Evaluate on:
        - **Readability**: Naming, formatting, comments
        - **Maintainability**: Structure, modularity, duplication
        - **Linting**: Use get_linting_score tool
        - **Best Practices**: README, .gitignore, requirements.txt

        Provide analysis (2-3 sentences) and score (0-100%) for each.
        Present in Markdown. Do not output raw file contents.
        """,
    tools=[
        FunctionTool(read_file),
        FunctionTool(read_directory_files),
        FunctionTool(read_github_repository),
        FunctionTool(get_linting_score),
        FunctionTool(cleanup_temp_directory),
    ],
)

# AGENT 3: Description Generator
description_generator = LlmAgent(
    model=Gemini(model="gemini-3.5-flash-lite", retry_options=retry_config),
    name="description_generator",
    description="Generates comprehensive code descriptions",
    instruction="""
        You are an expert at code documentation. Analyze and describe:
        1. **Overview**: High-level purpose, architecture, languages/libraries
        2. **Component Breakdown**: Key files, classes, functions, their roles
        3. **Functionality Summary**: Concise one-line summary

        Use Markdown formatting. Do not include raw code.
        """,
    tools=[
        FunctionTool(read_file),
        FunctionTool(read_directory_files),
        FunctionTool(read_github_repository),
        FunctionTool(get_linting_score),
        FunctionTool(cleanup_temp_directory),
    ],
)

# AGENT 4: Improvement Recommender
improvement_recommender = LlmAgent(
    model=Gemini(model="gemini-3.5-flash-lite", retry_options=retry_config),
    name="improvement_recommender",
    tools=[
        AgentTool(description_generator),
        AgentTool(correctness_assessor),
        AgentTool(style_assessor)
    ],
    description="Suggests actionable code improvements",
    instruction="""
        Analyze assessments from other agents and recommend improvements:
        1. **Functionality & Correctness**: Bugs, error handling, security
        2. **Code Style & Quality**: Readability, maintainability, linting
        3. **Best Practices**: Documentation, configuration files

        Present as clear, actionable Markdown list.
        """,
)

# AGENT 5: Report Generator (Root Agent)
report_generator = LlmAgent(
    model=Gemini(model="gemini-3.5-flash-lite", retry_options=retry_config),
    name="report_generator",
    description="Compiles comprehensive assessment report",
    tools=[
        AgentTool(description_generator),
        AgentTool(correctness_assessor),
        AgentTool(style_assessor),
        AgentTool(improvement_recommender),
        preload_memory
    ],
    instruction="""
        Compile outputs from all agents into a structured report:

        # Code Assessment Report
        ## 1. Code Description
        ## 2. Code Correctness Assessment
        ## 3. Code Style Assessment
        ## 4. Suggested Improvements

        **CRITICAL**: Provide a complete written text report. 
        Do not just call tools without synthesizing results.
        """,
)

## 6️⃣ Browser Display Utility

This function converts the markdown report to a beautifully styled HTML page with:
- Professional typography and layout
- Markdown rendering via markdown-it library
- Syntax highlighting for code blocks
- Responsive design

In [6]:
def display_report_in_browser(report_text: str, filename="code_assessment_report.html"):
    """Converts markdown report to styled HTML and opens in browser."""
    import webbrowser
    
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Code Assessment Report</title>
        <style>
            body {{
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
                line-height: 1.6;
                max-width: 900px;
                margin: 0 auto;
                padding: 20px;
                background: #f5f5f5;
            }}
            .container {{
                background: white;
                padding: 40px;
                border-radius: 8px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            }}
            h1 {{
                color: #2c3e50;
                border-bottom: 3px solid #3498db;
                padding-bottom: 10px;
            }}
            h2 {{
                color: #34495e;
                margin-top: 30px;
                border-bottom: 2px solid #ecf0f1;
                padding-bottom: 8px;
            }}
            pre {{
                background-color: #f8f9fa;
                border-left: 4px solid #3498db;
                padding: 15px;
                border-radius: 4px;
                overflow-x: auto;
            }}
            code {{
                background-color: #f8f9fa;
                padding: 2px 6px;
                border-radius: 3px;
            }}
        </style>
        <script src="https://cdn.jsdelivr.net/npm/markdown-it@13/dist/markdown-it.min.js"></script>
    </head>
    <body>
        <div class="container">
            <div id="content"></div>
        </div>
        <script>
            var md = window.markdownit();
            var markdownText = `{report_text.replace('`', '\\`').replace('$', '\\$')}`;
            document.getElementById('content').innerHTML = md.render(markdownText);
        </script>
    </body>
    </html>
    """
    
    reports_dir = Path("./reports")
    reports_dir.mkdir(exist_ok=True)
    file_path = reports_dir / filename
    
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print(f"📄 Report saved to: {file_path}")
    webbrowser.open(f'file://{file_path.absolute()}')
    print(f"🌐 Opening report in browser...")
    
    return str(file_path)

## 7️⃣ Main Execution Function

Configure your code assessment by uncommenting one of the query options:
- **Single file**: Analyze a specific Python file
- **Directory**: Analyze an entire project directory
- **Repository**: Clone and analyze a GitHub repository

The workflow:
1. Initialize ADK environment
2. Create multi-agent application
3. Process assessment request
4. Display results in notebook and browser

In [7]:
async def main():
    """Main execution function for code assessment workflow."""
    
    # ========== CONFIGURATION ==========
    # Choose ONE of the following options:
    
    code_file_path = "/home/samer/Documents/studies/5-Day AI Agents Intensive kaggle/capstone_project/agents_intensive_dev/src/llm_adk_agent_module.py"
    # code_dir_path = "/path/to/your/directory"
    # repo_http_path = "https://github.com/AI-Hypercomputer/maxtext.git"
    
    USER_ID = "explorer"
    session_id = "explorer_session"
    # ===================================

    # Create multi-agent application
    code_broker_app = App(
        name="code_broker",
        root_agent=report_generator,
        resumability_config=ResumabilityConfig(is_resumable=False),
    )

    # Initialize environment
    credentials = await initialize_adk_model()
    session_service = InMemorySessionService()
    memory_service = InMemoryMemoryService()

    # Create or retrieve session
    try:
        session = await session_service.create_session(
            session_id=session_id,
            app_name="code_broker",
            user_id=USER_ID
        )
    except:
        session = await session_service.get_session(
            session_id=session_id,
            app_name="code_broker",
            user_id=USER_ID
        )

    # Create runner
    coding_runner = Runner(
        app=code_broker_app,
        session_service=session_service,
        memory_service=memory_service,
    )
    
    # Construct request (uncomment the one you want to use)
    query = [f"provide a report for the code file:{code_file_path}"]
    # query = [f"provide a report for the directory:{code_dir_path}"]
    # query = [f"provide a report for the repository at:{repo_http_path}"]
    
    user_message = types.Content(role="user", parts=[types.Part(text=query[0])])

    report_text = ""
    
    print("🚀 Starting code assessment...")
    print("⏳ This may take a few minutes...\n")
    
    # Execute agent workflow
    async for event in coding_runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=user_message
    ):
        if event.is_final_response() and event.content:
            if hasattr(event.content, 'parts') and event.content.parts:
                text_parts = [
                    p.text for p in event.content.parts
                    if hasattr(p, 'text') and p.text
                ]
                if text_parts:
                    report_text = '\n'.join(text_parts)
                    if report_text and report_text != "None":
                        # Display in notebook
                        display(Markdown("## 📊 Code Assessment Report"))
                        display(Markdown("---"))
                        display(Markdown(report_text))
                        
                        # Open in browser
                        print("\n" + "="*60)
                        display_report_in_browser(report_text)
                        print("="*60)
            else:
                print("⚠️  Warning: No text content in response")

    print("\n✅ Assessment complete!")

# Run the assessment
if __name__ == "__main__":
    await main()

/tmp/ipykernel_133568/128405431.py:19: UserWarning: [EXPERIMENTAL] ResumabilityConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  resumability_config=ResumabilityConfig(is_resumable=False),


🚀 Starting code assessment...
⏳ This may take a few minutes...



/home/samer/anaconda3/envs/agents/lib/python3.14/site-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


## 📊 Code Assessment Report

---

# Code Assessment Report
## 1. Code Description
The `llm_adk_agent_module.py` module is an asynchronous Python application that initializes and executes a specialized LLM agent using the **Google Agent Development Kit (ADK)**. Its primary role is to act as an expert programming assistant capable of code analysis, quality assessment, feature integration suggestions, resource optimization, and repository documentation using the `gemini-2.0-flash` model combined with custom toolsets.

### Component Breakdown:
* **Imports & Configuration**: Uses `os`, `asyncio`, `dotenv`, and Google ADK components (`LlmAgent`, `InMemoryRunner`, `FunctionTool`). Integrates local file, lint, and GitHub repository utilities.
* **`initialize_adk_model() -> LlmAgent`**: Validates environment variables (`GOOGLE_API_KEY`, `GITHUB_TOKEN`) and initializes an `LlmAgent` named `"helpful_assistant"` with the `gemini-2.0-flash` model, developer instructions, and registered `FunctionTool`s.
* **`generate_adk_response(prompt: str) -> str`**: Asynchronously initializes the agent, creates an `InMemoryRunner`, runs a debug session against the provided prompt, and returns the response.
* **`main()`**: Entry point demonstrating asynchronous usage of `generate_adk_response` with an example prompt analyzing a target GitHub repository.

---

## 2. Code Correctness Assessment
* **Functionality**: The code correctly sets up environment variables, initializes the ADK agent, registers tools, and executes queries via `InMemoryRunner`. However, there is an environment variable name discrepancy (`GOOGLE_API_KEY` checked vs. `"GEMINI_API_KEY environment variable not set."` error message) and a truncated phrase in the `main()` function prompt. (**Score: 85%**)
* **Security**: Sensitive credentials (`GOOGLE_API_KEY`, `GITHUB_TOKEN`) are correctly retrieved via environment variables. Ensure no credential leaks occur in logs. (**Score: 90%**)
* **Resource Efficiency**: Uses asynchronous programming (`asyncio`), but re-instantiates the agent and runner on every call to `generate_adk_response` rather than reusing them. (**Score: 80%**)
* **Test Coverage**: Currently lacks unit or integration tests, relying solely on manual script execution in `main()`. (**Score: 0%**)

---

## 3. Code Style Assessment
* **Readability**: Follows standard Python PEP 8 formatting conventions with clear naming and docstrings. (**Score: 85/100**)
* **Maintainability**: Clear separation of concerns between initialization, execution, and entry point logic, though further decoupling of env configuration would improve flexibility. (**Score: 80/100**)
* **Linting**: Syntactically clean and properly structured for asynchronous execution. (**Score: 90/100**)
* **Best Practices**: Correctly uses `python-dotenv` for managing sensitive keys, though a root-level `.env.example` template is recommended. (**Score: 85/100**)

---

## 4. Suggested Improvements
1. **Fix Prompt Typo & Incompleteness**: Complete the truncated instruction in the `main()` function prompt (e.g., change `"provide a for the repository"` to `"provide a comprehensive report for the repository"`).
2. **Align Environment Variable Names**: Ensure consistency between the environment variable being checked (`GOOGLE_API_KEY`) and the exception message or variable name (`GEMINI_API_KEY`).
3. **Optimize Agent Lifecycle**: Avoid instantiating a new agent and `InMemoryRunner` on every call to `generate_adk_response()` by caching or maintaining persistent instances.
4. **Sort and Clean Imports**: Group and sort standard library imports before third-party packages in accordance with PEP 8 standards.
5. **Implement Unit Tests**: Add a test suite (`pytest` with `pytest-asyncio`) to mock tool calls and ADK execution without making live API requests.
6. **Add Configuration Template**: Include a `.env.example` file to document necessary environment variables (`GOOGLE_API_KEY`, `GITHUB_TOKEN`).


📄 Report saved to: reports/code_assessment_report.html
🌐 Opening report in browser...

✅ Assessment complete!


## 📚 Usage Instructions

### Prerequisites
1. Create a `.env` file in your project root:
   ```
   GOOGLE_API_KEY=your_gemini_api_key_here
   GITHUB_TOKEN=your_github_token_here  # Optional
   ```

2. Install dependencies:
   ```bash
   pip install google-adk python-dotenv pylint
   ```

### Running the Assessment

1. **Configure** the `main()` function with your target:
   - Update `code_file_path`, `code_dir_path`, or `repo_http_path`
   - Uncomment the appropriate `query` line

2. **Execute** all cells from top to bottom

3. **View Results**:
   - Report preview in notebook
   - Full HTML report opens in browser
   - Saved to `./reports/code_assessment_report.html`

### Troubleshooting

- **API Key Error**: Ensure `.env` file exists with valid credentials
- **Pylint Not Found**: Install with `pip install pylint`
- **Rate Limiting**: Retry configuration handles this automatically
- **No Text Response**: Check that agents have proper tool access
- **Linting Unavailable**: Provide the path of the file or directory in quotations


## 8️⃣ Session Memory Storage (Optional)

### Store Assessment Results in Memory

This optional cell demonstrates how to persist the agent session to memory. This allows:
- **Session Persistence**: Save conversation history and agent states
- **Memory Retrieval**: Access previous assessment results
- **Context Preservation**: Maintain context across multiple assessments

**Use Case**: When running multiple assessments, you can reference previous results and compare findings.

**Note**: Uncomment this cell if you want to enable memory storage.

In [8]:
# Optional: Store session to memory for future retrieval
# Uncomment the lines below to enable memory storage

# exploring_session = await session_service.get_session(
#     app_name="code_broker", user_id=USER_ID, session_id=session_id
# )

# await memory_service.add_session_to_memory(exploring_session)
# print("✅ Session stored to memory successfully!")

## 9️⃣ Memory Search and Retrieval (Optional)

### Search Previous Assessment Results

This cell demonstrates the ADK's memory search capability:
- **Semantic Search**: Find relevant information from past assessments
- **Query-Based Retrieval**: Ask questions about previous reports
- **Context Awareness**: Access historical assessment data

**Example Queries:**
- "What are previous reports about?"
- "What were the security findings?"
- "What code style issues were found?"

**Prerequisites**: 
- Session must be stored to memory (see previous cell)
- At least one assessment must have been completed

**Note**: This is useful for tracking improvements over time or comparing different codebases.

In [9]:
# Optional: Search through stored assessment memories
# Uncomment the lines below to enable memory search

# search_response = await memory_service.search_memory(
#     app_name="code_broker", 
#     user_id=USER_ID, 
#     query="What are previous reports about?"  # Modify this query as needed
# )

# print("🔍 Search Results:")
# print(f"  Found {len(search_response.memories)} relevant memories")
# print()

# for memory in search_response.memories:
#     if memory.content and memory.content.parts:
#         text = memory.content.parts[0].text[:80]
#         print(f"  [{memory.author}]: {text}...")

# # Display full memory content if needed
# if search_response.memories:
#     print("\n📄 Full Memory Content:")
#     for i, memory in enumerate(search_response.memories, 1):
#         if memory.content and memory.content.parts:
#             full_text = memory.content.parts[0].text
#             print(f"\n--- Memory {i} ---")
#             display(Markdown(full_text[:500]))  # Show first 500 chars

/usr/share/antigravity/antigravity: bad option: --no-sandbox


## 🚀 Advanced Features and Next Steps

### Additional Capabilities

#### **1. Batch Processing**
Run assessments on multiple files or repositories:
```python
files = ["file1.py", "file2.py", "file3.py"]
for file_path in files:
    # Run assessment for each file
    # Store results with unique session IDs
```

#### **2. Comparative Analysis**
Compare assessment results across versions:
```python
# Assess version 1
# Store to memory with session_id="v1_assessment"

# Assess version 2
# Store to memory with session_id="v2_assessment"

# Search memory to compare findings
```

#### **3. Custom Agent Configuration**
Modify agent instructions for specialized assessments:
- **Security-focused**: Emphasize vulnerability detection
- **Performance-focused**: Prioritize efficiency metrics
- **Documentation-focused**: Focus on code clarity and comments
- **Research Focused**: Add google search tool to provide more information about the code
- **Code test generator**: Generate test cases for the code
- **Code language translator**: Translate code to another language

#### **4. Export and Share**
- Save reports as PDF using HTML-to-PDF tools
- Export data for analytics dashboards
- Share reports with team members

### Best Practices

1. **Regular Assessments**: Run periodic code assessments during development
2. **Track Improvements**: Use memory to compare current vs. previous scores
3. **Team Integration**: Include reports in code review processes
4. **Continuous Learning**: Adjust agent instructions based on findings

### Resources

- [Google ADK Documentation](https://google.github.io/adk-docs/)
- [Gemini API Reference](https://ai.google.dev/docs)
- [Pylint Documentation](https://pylint.pycqa.org/)

---

**Happy Coding! 🎉**